In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

#### Load the data

In [2]:
%pip install ucimlrepo


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

### Explore the Data

In [4]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

((569, 30),
 (569,),
 None,
 None,
 Diagnosis
 B    357
 M    212
 Name: count, dtype: int64)

#### Check for duplicates

In [5]:
X.duplicated().sum()

np.int64(0)

In [6]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)

### Load the shared agentic data partition

Use the saved Level 1 agentic partition manifest for the 455 development rows and reserved 114-row test set. The conventional development evaluation then loads the saved outer split manifest and evaluates the exact same train and validation row IDs.


In [7]:
from pathlib import Path
import json
import os

def _find_agentic_reference_dir():
    explicit_dir = os.environ.get("AGENTIC_REFERENCE_DIR")
    if explicit_dir:
        candidate = Path(explicit_dir).expanduser().resolve()
        if (candidate / "data_partition_manifest.json").is_file() and (candidate / "split_manifest.csv").is_file():
            return candidate
        raise FileNotFoundError(f"AGENTIC_REFERENCE_DIR is missing required manifest files: {candidate}")
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base / "Agentic" / "level_1", base / "level_1"])
    for candidate in candidates:
        if (candidate / "data_partition_manifest.json").is_file() and (candidate / "split_manifest.csv").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find Agentic/level_1/data_partition_manifest.json and split_manifest.csv. "
        "Open this notebook from the Conventional project or set AGENTIC_REFERENCE_DIR explicitly."
    )

AGENTIC_REFERENCE_DIR = _find_agentic_reference_dir()
partition_manifest = json.loads(
    (AGENTIC_REFERENCE_DIR / "data_partition_manifest.json").read_text(encoding="utf-8")
)
development_row_ids = [int(value) for value in partition_manifest["development_row_ids"]]
reserved_test_row_ids = [int(value) for value in partition_manifest["reserved_test_row_ids"]]

if set(development_row_ids) & set(reserved_test_row_ids):
    raise ValueError("Agentic development and reserved test row IDs overlap.")
if set(development_row_ids) | set(reserved_test_row_ids) != set(X.index):
    raise ValueError("Agentic partition IDs do not match the loaded dataset row IDs.")

# Preserve the manifest's row order and keep the final test partition out of development selection.
X_train = X.loc[development_row_ids].copy()
y_train = y.loc[development_row_ids].copy()
X_test = X.loc[reserved_test_row_ids].copy()
y_test = y.loc[reserved_test_row_ids].copy()

print(f"Loaded shared agentic partition from: {AGENTIC_REFERENCE_DIR}")
print(f"Development rows: {len(X_train)} | Reserved test rows: {len(X_test)}")


Loaded shared agentic partition from: C:\Users\ogeuc\Downloads\Agentic\level_1
Development rows: 455 | Reserved test rows: 114


**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.

### Define models, apply transformation, select features and evaluate the models

**Define Candidate Model Pipelines**

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=42))
])

XGB_pipeline = Pipeline([
    ("clf", xgb.XGBClassifier(random_state=42))
])

LR_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

RF_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

SVC_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

### Shared outer holdouts with five-fold inner CV


In [9]:
import time
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

RANDOM_SEEDS = [11, 42, 67]
REPEATS_PER_SEED = 3
CV_FOLDS = 5
SELECTION_METRIC = "Recall primary, F1 secondary"
ML_MODELS = {"LR": LR_pipeline, "SVC": SVC_pipeline, "RF": RF_pipeline, "GB": GB_pipeline, "XGB": XGB_pipeline}

shared_splits_df = pd.read_csv(AGENTIC_REFERENCE_DIR / "split_manifest.csv")
required_split_columns = {"Run_ID", "Seed", "Repeat", "Development_Train_Row_IDs", "Outer_Validation_Row_IDs"}
missing_columns = required_split_columns - set(shared_splits_df.columns)
if missing_columns:
    raise ValueError(f"Agentic split manifest is missing columns: {sorted(missing_columns)}")
if len(shared_splits_df) != len(RANDOM_SEEDS) * REPEATS_PER_SEED:
    raise ValueError("Expected exactly three repeats for each of the three agentic seeds.")
if set(shared_splits_df["Seed"].astype(int)) != set(RANDOM_SEEDS):
    raise ValueError("Agentic split manifest seeds do not match the expected seed list.")

def _row_ids(value):
    return [int(row_id) for row_id in json.loads(value)]

shared_splits_df = shared_splits_df.sort_values(["Seed", "Repeat", "Run_ID"]).reset_index(drop=True)
shared_splits = []
development_ids = set(X_train.index)
for split in shared_splits_df.to_dict("records"):
    train_ids = _row_ids(split["Development_Train_Row_IDs"])
    validation_ids = _row_ids(split["Outer_Validation_Row_IDs"])
    if set(train_ids) & set(validation_ids):
        raise ValueError(f"Train/validation overlap in agentic run {split['Run_ID']}.")
    if set(train_ids) | set(validation_ids) != development_ids:
        raise ValueError(f"Run {split['Run_ID']} does not partition the shared development rows.")
    shared_splits.append({**split, "train_ids": train_ids, "validation_ids": validation_ids})

for seed in RANDOM_SEEDS:
    seed_splits = [row for row in shared_splits if int(row["Seed"]) == seed]
    if len(seed_splits) != REPEATS_PER_SEED:
        raise ValueError(f"Seed {seed} does not have exactly three repeats.")
    reference = (seed_splits[0]["train_ids"], seed_splits[0]["validation_ids"])
    if any((row["train_ids"], row["validation_ids"]) != reference for row in seed_splits[1:]):
        raise ValueError(f"Repeated rows for seed {seed} do not reuse identical agentic indices.")

def _recall_then_f1_refit(cv_results):
    recall_scores = np.asarray(cv_results["mean_test_recall"], dtype=float)
    f1_scores = np.asarray(cv_results["mean_test_f1"], dtype=float)
    k_values = np.asarray([params["selector__k"] for params in cv_results["params"]])
    best_recall = np.nanmax(recall_scores)
    recall_ties = np.flatnonzero(np.isclose(recall_scores, best_recall))
    best_f1 = np.nanmax(f1_scores[recall_ties])
    finalists = recall_ties[np.isclose(f1_scores[recall_ties], best_f1)]
    return int(finalists[np.argmin(k_values[finalists])])

def feature_search(model, X_fit, y_fit, cv):
    pipeline = Pipeline([
        ("selector", SelectKBest(score_func=f_classif)),
        ("classifier", clone(model)),
    ])
    return GridSearchCV(
        estimator=pipeline,
        param_grid={"selector__k": list(range(1, X_fit.shape[1] + 1))},
        scoring={"recall": "recall", "f1": "f1"},
        refit=_recall_then_f1_refit,
        cv=cv, n_jobs=-1, return_train_score=False,
    )

def _best_search_metrics(search):
    ix = int(search.best_index_)
    return {
        "Inner_CV_Recall": float(search.cv_results_["mean_test_recall"][ix]),
        "Inner_CV_Recall_SD": float(search.cv_results_["std_test_recall"][ix]),
        "Inner_CV_F1": float(search.cv_results_["mean_test_f1"][ix]),
        "Inner_CV_F1_SD": float(search.cv_results_["std_test_f1"][ix]),
        "Selected_k": int(search.best_params_["selector__k"]),
        "Candidate_Configurations": len(search.cv_results_["params"]),
        "CV_Fits": len(search.cv_results_["params"]) * search.n_splits_ + 1,
    }

def score_validation(model, X_validation, y_validation):
    y_pred = model.predict(X_validation)
    if hasattr(model, "predict_proba"):
        positive_index = list(model.classes_).index(1)
        y_score = model.predict_proba(X_validation)[:, positive_index]
    else:
        y_score = model.decision_function(X_validation)
        if model.classes_[1] != 1:
            y_score = -y_score
    return {
        "Accuracy": accuracy_score(y_validation, y_pred),
        "Precision": precision_score(y_validation, y_pred, zero_division=0),
        "Recall": recall_score(y_validation, y_pred, zero_division=0),
        "F1": f1_score(y_validation, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_validation, y_score),
        "Confusion_matrix": confusion_matrix(y_validation, y_pred, labels=[0, 1]).tolist(),
    }


#### Inner feature selection uses five-fold CV; outer scoring uses the shared holdout rows


In [10]:
# `feature_search` above ranks k by recall, then F1, with smaller k as a final deterministic tie-break.


In [11]:
candidate_cv_rows = []
repeated_results = []
for split in shared_splits:
    run_id, seed, repeat = int(split["Run_ID"]), int(split["Seed"]), int(split["Repeat"])
    X_outer_train = X.loc[split["train_ids"]]
    y_outer_train = y.loc[split["train_ids"]]
    X_outer_validation = X.loc[split["validation_ids"]]
    y_outer_validation = y.loc[split["validation_ids"]]
    inner_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    searches = {}
    run_candidate_rows = []
    run_started = time.perf_counter()

    for model_name, model in ML_MODELS.items():
        search = feature_search(model, X_outer_train, y_outer_train, inner_cv)
        search_started = time.perf_counter()
        search.fit(X_outer_train, y_outer_train)
        search_runtime = time.perf_counter() - search_started
        searches[model_name] = search
        details = _best_search_metrics(search)
        candidate_row = {
            "Run_ID": run_id, "Seed": seed, "Repeat": repeat, "Model": model_name,
            "Runtime_seconds": float(search_runtime),
            **details,
        }
        candidate_cv_rows.append(candidate_row)
        run_candidate_rows.append(candidate_row)

    ranked = pd.DataFrame(run_candidate_rows).sort_values(
        ["Inner_CV_Recall", "Inner_CV_F1"], ascending=[False, False], kind="stable"
    )
    selected = ranked.iloc[0]
    selected_model_name = selected["Model"]
    selected_search = searches[selected_model_name]
    outer_metrics = score_validation(selected_search.best_estimator_, X_outer_validation, y_outer_validation)
    total_runtime = time.perf_counter() - run_started
    repeated_results.append({
        "Level": "Conventional feature selection", "Run_ID": run_id, "Seed": seed, "Repeat": repeat,
        "Split_Seed": seed, "Status": "completed", "Classifier": selected_model_name,
        "Feature_Method": "SelectKBest(f_classif)", "Selected_k": int(selected["Selected_k"]),
        "Selection_Metric": SELECTION_METRIC,
        "Inner_CV_Recall": float(selected["Inner_CV_Recall"]), "Inner_CV_F1": float(selected["Inner_CV_F1"]),
        "CV_Folds": CV_FOLDS, "Model_Evaluations": int(sum(row["Candidate_Configurations"] for row in run_candidate_rows)),
        "Model_Fits": int(sum(row["CV_Fits"] for row in run_candidate_rows)),
        "Agent_Runtime": None, "Total_Run_Wall_Time": total_runtime,
        **{f"Outer_Validation_{key}": value for key, value in outer_metrics.items() if key != "Confusion_matrix"},
        "Outer_Validation_Confusion_Matrix": json.dumps(outer_metrics["Confusion_matrix"]),
        "Development_Train_Row_IDs": json.dumps(split["train_ids"]),
        "Outer_Validation_Row_IDs": json.dumps(split["validation_ids"]),
    })

candidate_cv_df = pd.DataFrame(candidate_cv_rows)
repeated_results_df = pd.DataFrame(repeated_results)


In [12]:
candidate_summary_df = (candidate_cv_df.groupby("Model", as_index=False)
    .agg(Inner_CV_Recall_Mean=("Inner_CV_Recall", "mean"), Inner_CV_Recall_SD=("Inner_CV_Recall", "std"),
         Inner_CV_F1_Mean=("Inner_CV_F1", "mean"), Inner_CV_F1_SD=("Inner_CV_F1", "std"),
         Selected_k_Mean=("Selected_k", "mean"), Selected_k_SD=("Selected_k", "std"),
         Runtime_Seconds_Mean=("Runtime_seconds", "mean"), Runtime_Seconds_SD=("Runtime_seconds", "std"))
    .sort_values(["Inner_CV_Recall_Mean", "Inner_CV_F1_Mean"], ascending=[False, False]).reset_index(drop=True))
outer_metrics = ["Outer_Validation_Accuracy", "Outer_Validation_Precision", "Outer_Validation_Recall", "Outer_Validation_F1", "Outer_Validation_AUC"]
outer_summary_df = pd.DataFrame([{
    "Outer_Runs": len(repeated_results_df),
    **{f"{metric}_Mean": float(repeated_results_df[metric].mean()) for metric in outer_metrics},
    **{f"{metric}_SD": float(repeated_results_df[metric].std(ddof=1)) for metric in outer_metrics},
    "Wall_Time_Seconds_Mean": float(repeated_results_df["Total_Run_Wall_Time"].mean()),
    "Candidate_Configurations_Mean": float(repeated_results_df["Model_Evaluations"].mean()),
    "Model_Fits_Mean": float(repeated_results_df["Model_Fits"].mean()),
}])
selection_consistency_df = (repeated_results_df.groupby("Classifier", as_index=False)
    .agg(Run_Count=("Run_ID", "count"), Seeds=("Seed", lambda values: sorted(set(map(int, values)))))
    .sort_values("Run_Count", ascending=False).reset_index(drop=True))
display(candidate_summary_df.round(4))
display(outer_summary_df.round(4))
display(selection_consistency_df)


,Model,Inner_CV_Recall_Mean,Inner_CV_Recall_SD,Inner_CV_F1_Mean,Inner_CV_F1_SD,Selected_k_Mean,Selected_k_SD,Runtime_Seconds_Mean,Runtime_Seconds_SD
0,XGB,0.9558,0.0128,0.9593,0.0039,19.3333,1.0000,9.1212,0.5394
1,LR,0.9534,0.0096,0.9651,0.0066,23.0000,3.1225,1.5625,2.2447
2,SVC,0.9533,0.0075,0.9638,0.0048,25.0000,3.9686,0.7602,0.1162
3,GB,0.9360,0.0038,0.9434,0.0034,23.3333,5.5678,4.8511,0.5461
4,RF,0.9312,0.0038,0.9430,0.0046,24.0000,1.5000,4.3780,0.1926


,Outer_Runs,Outer_Validation_Accuracy_Mean,Outer_Validation_Precision_Mean,Outer_Validation_Recall_Mean,Outer_Validation_F1_Mean,Outer_Validation_AUC_Mean,Outer_Validation_Accuracy_SD,Outer_Validation_Precision_SD,Outer_Validation_Recall_SD,Outer_Validation_F1_SD,Outer_Validation_AUC_SD,Wall_Time_Seconds_Mean,Candidate_Configurations_Mean,Model_Fits_Mean
0,9,0.9707,0.9705,0.951,0.9603,0.9964,0.0145,0.0248,0.0294,0.02,0.0022,20.7226,150.0,755.0


,Classifier,Run_Count,Seeds
0,XGB,6,"[11, 67]"
1,SVC,3,[42]


In [13]:
# Candidate and outer-holdout summaries are displayed in the preceding cell.


In [14]:
print("Outer holdout runs:", len(repeated_results_df))
print("Seeds:", RANDOM_SEEDS, "| repeats per seed:", REPEATS_PER_SEED)
print("Selection rule: highest inner-CV recall, then F1; smaller k breaks remaining ties.")


Outer holdout runs: 9
Seeds: [11, 42, 67] | repeats per seed: 3
Selection rule: highest inner-CV recall, then F1; smaller k breaks remaining ties.


In [15]:
# Select the final model and feature count on all development data only; do not read X_test or y_test here.
final_selection_rows = []
final_searches = {}
final_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=101)
final_fit_start = time.perf_counter()
final_fit_cpu_start = time.process_time()
for model_name, model in ML_MODELS.items():
    search = feature_search(model, X_train, y_train, final_cv)
    search.fit(X_train, y_train)
    final_searches[model_name] = search
    final_selection_rows.append({"Model": model_name, **_best_search_metrics(search)})
final_model_selection_df = pd.DataFrame(final_selection_rows).sort_values(
    ["Inner_CV_Recall", "Inner_CV_F1"], ascending=[False, False], kind="stable"
).reset_index(drop=True)
best_model_name = final_model_selection_df.iloc[0]["Model"]
final_search = final_searches[best_model_name]
final_model = final_search.best_estimator_
final_k = int(final_search.best_params_["selector__k"])
selected_features = X_train.columns[final_model.named_steps["selector"].get_support()].tolist()
final_development_fit_seconds = time.perf_counter() - final_fit_start
final_fit_cpu_seconds = time.process_time() - final_fit_cpu_start

folder_name = "eval - feature selection"
output_dir = Path.cwd() if Path.cwd().name == folder_name else Path.cwd() / folder_name
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "conventional_frozen_model.joblib"
model_save_start = time.perf_counter(); joblib.dump(final_model, model_path); model_save_seconds = time.perf_counter() - model_save_start
model_load_start = time.perf_counter(); loaded_best_pipeline = joblib.load(model_path); model_load_seconds = time.perf_counter() - model_load_start
final_workflow_wall_seconds = final_development_fit_seconds + model_save_seconds + model_load_seconds
selected_final_cv = final_model_selection_df.iloc[0]
best_record = {
    "Best_Model": best_model_name, "Selection_Metric": SELECTION_METRIC,
    "Random_Seeds": RANDOM_SEEDS, "CV_Folds": CV_FOLDS,
    "Repeated_Outer_Holdouts_Per_Seed": REPEATS_PER_SEED, "Outer_Run_Count": len(shared_splits),
    "Final_Selected_Feature_Count": final_k, "Selected_Features": selected_features,
    "Final_Development_CV_Recall": float(selected_final_cv["Inner_CV_Recall"]),
    "Final_Development_CV_F1": float(selected_final_cv["Inner_CV_F1"]),
    "Final_Development_CV_Recall_SD": float(selected_final_cv["Inner_CV_Recall_SD"]),
    "Final_Development_CV_F1_SD": float(selected_final_cv["Inner_CV_F1_SD"]),
    "Final_Development_Fit_Runtime_Seconds": final_development_fit_seconds,
    "Final_Fit_CPU_Seconds": final_fit_cpu_seconds,
    "Model_Save_Runtime_Seconds": model_save_seconds, "Model_Load_Runtime_Seconds": model_load_seconds,
    "Final_Test_Scoring_Wall_Seconds": None, "Final_Test_Scoring_CPU_Seconds": None,
    "Final_Workflow_Wall_Seconds": final_workflow_wall_seconds,
    "Test_Scored": False, "Saved_Model_Path": str(model_path),
    "Shared_Agentic_Split_Manifest": str(AGENTIC_REFERENCE_DIR / "split_manifest.csv"),
}

shared_splits_df.to_csv(output_dir / "split_manifest.csv", index=False)
(output_dir / "data_partition_manifest.json").write_text(
    (AGENTIC_REFERENCE_DIR / "data_partition_manifest.json").read_text(encoding="utf-8"), encoding="utf-8")
for seed in RANDOM_SEEDS:
    repeated_results_df.loc[repeated_results_df["Seed"] == seed].to_csv(output_dir / f"conventional_seed_{seed}_runs.csv", index=False)
seed_selections = repeated_results_df[["Run_ID", "Seed", "Repeat", "Classifier", "Selected_k", "Inner_CV_Recall", "Inner_CV_F1",
    "Outer_Validation_Recall", "Outer_Validation_F1"]].rename(columns={"Classifier": "Selected_Model"})
seed_selections.to_csv(output_dir / "conventional_seed_selections.csv", index=False)
selection_consistency_df.to_csv(output_dir / "decision_consistency.csv", index=False)
candidate_summary_df.to_csv(output_dir / "all_pipelines_summary.csv", index=False)
candidate_summary_df.to_csv(output_dir / "conventional_pipeline_summary.csv", index=False)
candidate_cv_df.to_csv(output_dir / "candidate_inner_cv_results.csv", index=False)
final_model_selection_df.to_csv(output_dir / "final_model_selection_cv.csv", index=False)
outer_summary_df.to_csv(output_dir / "conventional_summary.csv", index=False)
outer_summary_df.to_csv(output_dir / "outer_holdout_summary.csv", index=False)
repeated_results_df.to_csv(output_dir / "conventional_repeated_runs.csv", index=False)

run_metadata = {
    "Workflow": "conventional_feature_selection_shared_outer_holdout_evaluation",
    "Random_Seeds": RANDOM_SEEDS, "CV_Folds": CV_FOLDS,
    "Repeated_Outer_Holdouts_Per_Seed": REPEATS_PER_SEED, "Outer_Run_Count": len(shared_splits),
    "Outer_Split_Source": str(AGENTIC_REFERENCE_DIR / "split_manifest.csv"),
    "Data_Partition_Source": str(AGENTIC_REFERENCE_DIR / "data_partition_manifest.json"),
    "Development_Row_Count": len(X_train), "Reserved_Test_Row_Count": len(X_test),
    "Selection_Metric": SELECTION_METRIC, "Feature_Count_Tie_Break": "smaller k",
    "Inner_CV_Uses_Outer_Training_Rows_Only": True, "Test_Scored": False,
}
(output_dir / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
frozen_workflow = {
    "Workflow_State": "development_fit_complete_test_pending", "Selected_Model": best_model_name,
    "Selected_Feature_Count": final_k, "Saved_Model_Path": str(model_path),
    "Selection_Rule": SELECTION_METRIC, "Random_Seeds": RANDOM_SEEDS,
    "Repeated_Outer_Holdouts_Per_Seed": REPEATS_PER_SEED,
    "Split_Manifest": str(output_dir / "split_manifest.csv"),
    "Shared_Agentic_Split_Manifest": str(AGENTIC_REFERENCE_DIR / "split_manifest.csv"),
    "Run_Metadata": str(output_dir / "run_metadata.json"),
}
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps(frozen_workflow, indent=2), encoding="utf-8")
(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Selected final classifier and k using development CV recall, then F1:", best_model_name, final_k)
print("Saved final development model:", model_path)
print("The reserved test set was not scored.")


Selected final classifier and k using development CV recall, then F1: XGB 18
Saved final development model: c:\Users\ogeuc\Downloads\Conventional\eval - feature selection\conventional_frozen_model.joblib
The reserved test set was not scored.


### Final Development Fit and Saved Model (test scoring pending)

The saved estimator was selected and fit using development data only.


### Final Test Set Evaluation (one-time, after the workflow is frozen)

This is the only cell that scores the reserved test set. Do not use its result to modify code, settings, features, thresholds, or model choices.


In [16]:
# Run this cell only when ready to evaluate the held-out test set.
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

loaded_best_pipeline = joblib.load(model_path)
test_wall_start = time.perf_counter()
test_cpu_start = time.process_time()
y_test_pred = loaded_best_pipeline.predict(X_test)
if hasattr(loaded_best_pipeline, "predict_proba"):
    class_index = list(loaded_best_pipeline.classes_).index(1)
    y_test_score = loaded_best_pipeline.predict_proba(X_test)[:, class_index]
else:
    y_test_score = loaded_best_pipeline.decision_function(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_auc = roc_auc_score(y_test, y_test_score)
test_scoring_cpu_seconds = time.process_time() - test_cpu_start
test_scoring_seconds = time.perf_counter() - test_wall_start

best_record.update({
    "Final_Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Final_Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Test_Scored": True,
    "Final_Test_Accuracy": test_accuracy,
    "Final_Test_Precision": test_precision,
    "Final_Test_Recall": test_recall,
    "Final_Test_F1": test_f1,
    "Final_Test_ROC_AUC": test_auc,
    "Final_Test_Confusion_Matrix": confusion_matrix(y_test, y_test_pred).tolist()
})
test_metrics = {
    "Model": best_model_name,
    "Test_Accuracy": test_accuracy,
    "Test_Precision": test_precision,
    "Test_Recall": test_recall,
    "Test_F1": test_f1,
    "Test_ROC_AUC": test_auc,
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds
}
pd.DataFrame([test_metrics]).to_csv(output_dir / "conventional_final_test.csv", index=False)
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps({
    "Workflow_State": "test_scoring_complete",
    "Selected_Model": best_model_name,
    "Saved_Model_Path": str(model_path),
    "Test_Metrics_Path": str(output_dir / "conventional_final_test.csv"),
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Random_Seeds": RANDOM_SEEDS
}, indent=2), encoding="utf-8")
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)

(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Selected pipeline:", best_model_name)
print("Selected feature count:", final_k)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test ROC AUC: {test_auc:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification report:\n", classification_report(y_test, y_test_pred, zero_division=0))


Selected pipeline: XGB
Selected feature count: 18
Test accuracy: 0.9649
Test precision: 1.0000
Test recall: 0.9048
Test F1: 0.9500
Test ROC AUC: 0.9967
Confusion matrix:
 [[72  0]
 [ 4 38]]

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

